# Activity 4.1 – GTSRB Traffic Sign Detection (TensorFlow / Keras)

**Framework:** TensorFlow 2.x + Keras  
**Dataset:** GTSRB – German Traffic Sign Recognition Benchmark (43 classes, ~50k images)  
**Goal:** >90% classification accuracy  
**Models compared:** Custom CNN · AlexNet (transfer) · EfficientNet-B0 (transfer)  
**Hyperparameter search:** Keras Tuner (GridSearch) + manual experiment grid

> ⚠️ **Enable GPU before running:** Runtime → Change runtime type → T4 GPU

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 1 – Install dependencies
# ─────────────────────────────────────────────────────────────
!pip install -q tensorflow keras-tuner scikeras scikit-learn matplotlib kaggle

# Download GTSRB from Kaggle
# 1) Upload your kaggle.json API key (Profile → Settings → API → Create New Token)
# 2) Uncomment and run:
# import os
# os.makedirs('/root/.config/kaggle', exist_ok=True)
# !cp kaggle.json /root/.config/kaggle/
# !chmod 600 /root/.config/kaggle/kaggle.json
# !kaggle datasets download -d meowmeowmeowmeowmeow/gtsrb-german-traffic-sign
# !unzip -q gtsrb-german-traffic-sign.zip -d ./gtsrb

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 2 – Imports
# ─────────────────────────────────────────────────────────────
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks, optimizers
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import keras_tuner as kt

print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))
# Enable mixed precision for faster training on GPU
tf.keras.mixed_precision.set_global_policy('mixed_float16')

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 3 – Configuration
# ─────────────────────────────────────────────────────────────
IMG_SIZE     = 32          # Resize all images to 32×32 px
NUM_CLASSES  = 43          # GTSRB: 43 traffic sign categories
BATCH_SIZE   = 64          # Samples per gradient update
EPOCHS       = 30          # Maximum training iterations
LR           = 1e-3        # Adam learning rate
DATASET_PATH = './gtsrb'   # Adjust to your extracted dataset path

sign_names = [
    'Speed 20','Speed 30','Speed 50','Speed 60','Speed 70',
    'Speed 80','End Speed 80','Speed 100','Speed 120','No passing',
    'No passing >3.5t','Right-of-way','Priority road','Yield','Stop',
    'No vehicles','No vehicles >3.5t','No entry','General caution','Dangerous curve left',
    'Dangerous curve right','Double curve','Bumpy road','Slippery road','Road narrows right',
    'Road work','Traffic signals','Pedestrians','Children crossing','Bicycles crossing',
    'Ice/snow','Wild animals','End restrictions','Turn right ahead','Turn left ahead',
    'Go straight','Go straight/right','Go straight/left','Keep right','Keep left',
    'Roundabout','End no passing','End no passing >3.5t'
]

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 4 – Load dataset from directory
# Assumes GTSRB extracted to ./gtsrb/Train/<class_id>/<images>
# ─────────────────────────────────────────────────────────────
def load_gtsrb(base_path, img_size):
    """
    Walks the Train/<class_id>/ directory structure,
    resizes all images to img_size×img_size, returns (X, y) arrays.
    """
    X, y = [], []
    train_path = os.path.join(base_path, 'Train')
    for class_id in range(NUM_CLASSES):
        class_dir = os.path.join(train_path, str(class_id))
        if not os.path.exists(class_dir):
            continue
        for fname in os.listdir(class_dir):
            try:
                img = Image.open(os.path.join(class_dir, fname)).convert('RGB')
                img = img.resize((img_size, img_size))  # Uniform size required by CNN
                X.append(np.array(img))
                y.append(class_id)
            except Exception:
                pass
    return np.array(X), np.array(y)

X, y = load_gtsrb(DATASET_PATH, IMG_SIZE)
print(f'Loaded: {X.shape[0]} images, shape {X.shape[1:]}')

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 5 – Preprocessing
# ─────────────────────────────────────────────────────────────
# Normalize pixel values [0,255] → [0.0,1.0] for stable gradient computation
X = X.astype('float32') / 255.0

# One-hot encode: integer label 5 → [0,0,0,0,0,1,0,...,0]
y_cat = to_categorical(y, NUM_CLASSES)

# Stratified split: keeps class proportions equal in train and val
X_train, X_val, y_train, y_val = train_test_split(
    X, y_cat, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {len(X_train)} | Val: {len(X_val)}')

# Data augmentation: artificially expand dataset and improve robustness
train_aug = ImageDataGenerator(
    rotation_range=15,         # Random rotation ±15° (tilted signs)
    width_shift_range=0.1,     # Horizontal shift up to 10%
    height_shift_range=0.1,    # Vertical shift up to 10%
    zoom_range=0.15,           # Random zoom in/out
    brightness_range=[0.8, 1.2], # Simulate varied lighting
    horizontal_flip=False,     # Signs must NOT be flipped (they're not symmetric)
    fill_mode='nearest'        # Fill newly created pixels using nearest neighbor
)
train_aug.fit(X_train)

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 6 – Custom CNN Architecture
# 3 Conv blocks with BatchNorm + Dropout + Dense head
# ─────────────────────────────────────────────────────────────
def build_custom_cnn(input_shape=(IMG_SIZE, IMG_SIZE, 3),
                     num_classes=NUM_CLASSES,
                     filters=(32, 64, 128),  # Number of filters per conv block
                     dropout=0.4,            # Dropout rate in FC layer
                     dense_units=256):
    """
    Custom CNN with configurable filters, dropout, and dense units.
    Parameters are exposed so GridSearch / KerasTuner can optimize them.
    """
    model = models.Sequential([
        layers.Input(shape=input_shape),
        
        # ── Conv Block 1: low-level feature detection ──
        layers.Conv2D(filters[0], (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),   # Normalize activations after each conv
        layers.Conv2D(filters[0], (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),     # Spatial downsampling: 32×32 → 16×16
        layers.Dropout(0.25),          # Randomly deactivate neurons during training
        
        # ── Conv Block 2: mid-level pattern detection ──
        layers.Conv2D(filters[1], (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv2D(filters[1], (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),     # 16×16 → 8×8
        layers.Dropout(0.25),
        
        # ── Conv Block 3: high-level sign structure detection ──
        layers.Conv2D(filters[2], (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),     # 8×8 → 4×4
        layers.Dropout(0.25),
        
        # ── Fully Connected Classification Head ──
        layers.Flatten(),              # Flatten: 4×4×128 = 2048 values → 1D vector
        layers.Dense(dense_units, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(dropout),       # Stronger regularization in dense layer
        
        # ── Output: 43 classes with softmax probability ──
        layers.Dense(num_classes, activation='softmax', dtype='float32'),
    ], name='CustomCNN')
    return model

model_preview = build_custom_cnn()
model_preview.summary()

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 7 – Standard training callbacks
# Reused for all models
# ─────────────────────────────────────────────────────────────
def get_callbacks(model_name):
    return [
        # Stop training if val_accuracy doesn't improve for 7 epochs
        callbacks.EarlyStopping(
            monitor='val_accuracy', patience=7,
            restore_best_weights=True, verbose=1
        ),
        # Halve learning rate when val_loss plateaus for 3 epochs
        callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5,
            patience=3, min_lr=1e-6, verbose=1
        ),
        # Persist best model weights during training
        callbacks.ModelCheckpoint(
            f'{model_name}_best.keras', monitor='val_accuracy',
            save_best_only=True, verbose=0
        )
    ]

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 8 – WHAT IS GRIDSEARCH?
#
# GridSearchCV is a systematic hyperparameter tuning method from scikit-learn.
# It tries EVERY combination in a parameter grid using cross-validation
# and returns the best-performing combination.
#
# Example: grid = {lr: [1e-3, 3e-4], dropout: [0.3, 0.5], batch: [32, 64]}
# → GridSearch trains and evaluates 2×2×2 = 8 models automatically.
#
# For Keras, we use scikeras.KerasClassifier as a wrapper to make Keras
# models compatible with scikit-learn's GridSearchCV.
#
# ⚠️ WARNING: GridSearch is SLOW for CNNs on large datasets like GTSRB.
#    With 8 combinations × 10 epochs = 80 training runs.
#    Recommended: use it with a SMALL subset (5k images) and few epochs (5-10).
#    For production, prefer Keras Tuner (Cell 9) which is GPU-accelerated.
# ─────────────────────────────────────────────────────────────
from scikeras.wrappers import KerasClassifier
from sklearn.model_selection import GridSearchCV

# Use a small subset to keep GridSearch fast
GRID_SUBSET = 5000
idx = np.random.choice(len(X_train), GRID_SUBSET, replace=False)
X_gs = X_train[idx]
y_gs = np.argmax(y_train[idx], axis=1)  # GridSearchCV expects integer labels, not one-hot

# Wrap the Keras model builder for scikit-learn compatibility
def build_for_gridsearch(learning_rate=1e-3, dropout=0.4, dense_units=256):
    """Function factory: takes hyperparams as args, returns compiled Keras model."""
    model = build_custom_cnn(dropout=dropout, dense_units=dense_units)
    model.compile(
        optimizer=optimizers.Adam(learning_rate=learning_rate),
        loss='sparse_categorical_crossentropy',  # Works with integer labels
        metrics=['accuracy']
    )
    return model

# Wrap with KerasClassifier (scikeras bridge)
keras_clf = KerasClassifier(
    model=build_for_gridsearch,
    epochs=8,          # Few epochs per candidate to keep search fast
    verbose=0
)

# Define the search grid: keys must match build_for_gridsearch() argument names
param_grid = {
    'model__learning_rate': [1e-3, 3e-4],   # Learning rate candidates
    'model__dropout':       [0.3, 0.5],      # Dropout candidates
    'batch_size':           [32, 64],         # Batch size candidates
}

# GridSearchCV: cv=3 means 3-fold cross-validation per combination
# Total runs = 2×2×2 combinations × 3 folds = 24 mini-training runs
grid_search = GridSearchCV(
    estimator=keras_clf,
    param_grid=param_grid,
    cv=3,                   # 3-fold cross validation
    scoring='accuracy',
    verbose=2,
    n_jobs=1                # Set to 1 when using GPU (GPU doesn't parallelize safely)
)

print('Starting GridSearch... (may take 15–30 min on GPU with subset)')
gs_result = grid_search.fit(X_gs, y_gs)

print(f'\nBest GridSearch Score: {gs_result.best_score_*100:.2f}%')
print(f'Best Parameters: {gs_result.best_params_}')

# Show all results sorted by mean test score
gs_df = pd.DataFrame(gs_result.cv_results_)
gs_df_sorted = gs_df[['param_model__learning_rate','param_model__dropout',
                        'param_batch_size','mean_test_score','std_test_score']]\
               .sort_values('mean_test_score', ascending=False)
print('\nAll GridSearch Results:')
print(gs_df_sorted.to_string(index=False))

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 9 – Keras Tuner: GPU-accelerated hyperparameter search
# Better alternative to GridSearchCV for deep learning
# Supports Grid, Random, Bayesian, and Hyperband strategies
# ─────────────────────────────────────────────────────────────

def build_tunable_model(hp):
    """
    hp: HyperParameters object from Keras Tuner.
    Use hp.Choice / hp.Int / hp.Float to define searchable parameters.
    """
    # Search space for each hyperparameter
    lr      = hp.Choice('learning_rate', [1e-3, 3e-4, 1e-4])  # 3 LR options
    dropout = hp.Float('dropout', min_value=0.25, max_value=0.5, step=0.05)  # continuous range
    f1      = hp.Choice('filters_block1', [32, 64])            # Conv block 1 filters
    f2      = hp.Choice('filters_block2', [64, 128])           # Conv block 2 filters
    dense   = hp.Choice('dense_units', [128, 256, 512])        # FC layer size
    
    model = build_custom_cnn(filters=(f1, f2, 128), dropout=dropout, dense_units=dense)
    model.compile(
        optimizer=optimizers.Adam(learning_rate=lr),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# GridSearch tuner: tries all combinations (like scikit-learn GridSearchCV but GPU-native)
tuner = kt.GridSearch(
    hypermodel=build_tunable_model,
    objective='val_accuracy',       # Maximize validation accuracy
    max_trials=20,                  # Cap at 20 combinations (prevent explosion)
    overwrite=True,
    directory='kt_results',
    project_name='gtsrb_grid'
)
tuner.search_space_summary()

# Run tuner search
tuner.search(
    train_aug.flow(X_train, y_train, batch_size=64),
    validation_data=(X_val, y_val),
    epochs=10,           # Fewer epochs per trial to keep search fast
    callbacks=[callbacks.EarlyStopping('val_accuracy', patience=3)],
    verbose=0
)

best_hps = tuner.get_best_hyperparameters(1)[0]
print('\nBest hyperparameters found by Keras Tuner:')
for k in ['learning_rate','dropout','filters_block1','filters_block2','dense_units']:
    print(f'  {k}: {best_hps.get(k)}')

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 10 – Train Custom CNN with best hyperparameters
# ─────────────────────────────────────────────────────────────
best_custom = tuner.hypermodel.build(best_hps)
best_custom.compile(
    optimizer=optimizers.Adam(best_hps.get('learning_rate')),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print('Training Custom CNN with best hyperparameters...')
h_custom = best_custom.fit(
    train_aug.flow(X_train, y_train, batch_size=BATCH_SIZE),
    steps_per_epoch=len(X_train) // BATCH_SIZE,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    callbacks=get_callbacks('custom_cnn'),
    verbose=1
)

best_custom = keras.models.load_model('custom_cnn_best.keras')
_, acc_custom = best_custom.evaluate(X_val, y_val, verbose=0)
print(f'Custom CNN Val Accuracy: {acc_custom*100:.2f}%')

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 11 – Transfer Learning: AlexNet-inspired & EfficientNet-B0
#
# Transfer learning: use weights pretrained on ImageNet (~1.2M images, 1000 classes)
# and fine-tune only the classification head for GTSRB (43 classes).
# This dramatically reduces training time and improves accuracy.
# ─────────────────────────────────────────────────────────────

# ── Model A: AlexNet-inspired (manual implementation) ──
# Keras doesn't have a built-in AlexNet, so we implement the key structure:
# large conv filters → smaller conv → FC layers
def build_alexnet_style(input_shape=(IMG_SIZE, IMG_SIZE, 3), num_classes=NUM_CLASSES):
    """AlexNet-inspired architecture adapted for 32×32 input (original was 227×227)."""
    return models.Sequential([
        layers.Input(shape=input_shape),
        # Layer 1: large 5×5 filters (AlexNet used 11×11 on 227px input)
        layers.Conv2D(64, (5,5), activation='relu', padding='same'),
        layers.BatchNormalization(),     # AlexNet used LRN; BatchNorm is the modern equivalent
        layers.MaxPooling2D(2, 2),
        # Layer 2: 3×3 filters
        layers.Conv2D(128, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),
        # Layers 3-5: three 3×3 conv without pooling
        layers.Conv2D(256, (3,3), activation='relu', padding='same'),
        layers.Conv2D(256, (3,3), activation='relu', padding='same'),
        layers.Conv2D(128, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D(2, 2),
        # FC layers with Dropout (AlexNet introduced Dropout for FC layers)
        layers.Flatten(),
        layers.Dense(1024, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(512, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax', dtype='float32'),
    ], name='AlexNet_style')

# ── Model B: EfficientNet-B0 with pretrained ImageNet weights ──
def build_efficientnet(input_shape=(IMG_SIZE, IMG_SIZE, 3), num_classes=NUM_CLASSES):
    """
    EfficientNet-B0: compound-scaled architecture.
    include_top=False removes the ImageNet 1000-class head.
    We freeze the backbone and train only the new head (feature extraction mode).
    """
    # Load pretrained backbone (no top classification layer)
    backbone = tf.keras.applications.EfficientNetB0(
        weights='imagenet',           # Use weights pretrained on ImageNet
        include_top=False,            # Remove original 1000-class head
        input_shape=input_shape
    )
    backbone.trainable = False        # Freeze backbone: only train the new head
    
    inputs = layers.Input(shape=input_shape)
    # EfficientNet expects inputs in [0, 255] (it handles normalization internally)
    x = inputs * 255.0
    x = backbone(x, training=False)   # training=False keeps BatchNorm frozen
    x = layers.GlobalAveragePooling2D()(x)  # Collapse spatial dims: H×W×C → C
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation='softmax', dtype='float32')(x)
    return keras.Model(inputs, outputs, name='EfficientNet_B0')

print('AlexNet-style model:')
build_alexnet_style().summary()
print('\nEfficientNet-B0 model:')
build_efficientnet().summary()

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 12 – Compare all models
# ─────────────────────────────────────────────────────────────
def compile_and_train(model, name, lr=LR):
    model.compile(
        optimizer=optimizers.Adam(learning_rate=lr),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    history = model.fit(
        train_aug.flow(X_train, y_train, batch_size=BATCH_SIZE),
        steps_per_epoch=len(X_train) // BATCH_SIZE,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        callbacks=get_callbacks(name),
        verbose=1
    )
    best = keras.models.load_model(f'{name}_best.keras')
    _, val_acc = best.evaluate(X_val, y_val, verbose=0)
    return best, history, val_acc

results = []
histories = {}

# Train all models
for mname, mfn in [('alexnet',       build_alexnet_style),
                    ('efficientnet',  build_efficientnet)]:
    print(f'\nTraining {mname}...')
    m, h, acc = compile_and_train(mfn(), mname)
    histories[mname] = h
    param_count = m.count_params()
    results.append({'Model': mname, 'Parameters': f'{param_count/1e6:.1f}M',
                    'Val Accuracy': f'{acc*100:.2f}%',
                    '>90%': '✅' if acc > 0.9 else '❌'})

# Include Custom CNN result
_, acc_c = best_custom.evaluate(X_val, y_val, verbose=0)
results.insert(0, {'Model': 'custom_cnn', 'Parameters': f'{best_custom.count_params()/1e6:.1f}M',
                   'Val Accuracy': f'{acc_c*100:.2f}%', '>90%': '✅' if acc_c > 0.9 else '❌'})
histories['custom_cnn'] = h_custom

df_results = pd.DataFrame(results)
print('\n' + '='*55)
print('MODEL COMPARISON')
print(df_results.to_string(index=False))

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 13 – Plot training curves
# ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors = ['#2196F3', '#FF5722', '#4CAF50']

for (name, h), color in zip(histories.items(), colors):
    axes[0].plot(h.history['val_accuracy'], label=name, color=color, linewidth=2)
    axes[1].plot(h.history['val_loss'],     label=name, color=color, linewidth=2)

axes[0].axhline(0.90, color='red', linestyle='--', linewidth=1.5, label='90% target')
axes[0].set_title('Validation Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_title('Validation Loss')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('GTSRB – TF/Keras Model Comparison', fontsize=14)
plt.tight_layout()
plt.savefig('tf_training_curves.png', dpi=150)
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 14 – Export best model for Webots
# ─────────────────────────────────────────────────────────────
best_row = df_results.sort_values('Val Accuracy', ascending=False).iloc[0]
best_model_name = best_row['Model']
best_final_model = keras.models.load_model(f'{best_model_name}_best.keras')

# Option A: Keras native format (recommended)
best_final_model.save('gtsrb_best_model.keras')

# Option B: TFLite (lightweight, fast inference – best for Webots controller)
converter = tf.lite.TFLiteConverter.from_keras_model(best_final_model)
tflite_model = converter.convert()
with open('gtsrb_model.tflite', 'wb') as f:
    f.write(tflite_model)

# Save label map
with open('sign_names.json', 'w') as f:
    json.dump(sign_names, f, indent=2)

# Save results for report
df_results.to_csv('tf_model_comparison.csv', index=False)
gs_df_sorted.to_csv('tf_gridsearch_results.csv', index=False)

print(f'Best model: {best_model_name} – exported as .keras and .tflite')
print('Files ready for Webots controller integration.')